# V-JEPA 2 perception — train BC head & generate demo

Uses the cache built by `colab_smoke_test.ipynb` (now persisted on your Drive).

Pipeline:
1. Mount Drive
2. Pull latest code
3. Train a tiny BC head on the cached features
4. Generate a `demo.png` of predicted vs ground-truth actions

Total compute: ~1 minute on T4 (or CPU).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Pull latest code

In [ ]:
%cd /content
!git clone -b feat/vjepa2-perception https://github.com/PoCInnovation/LeWM-Robot.git || (cd LeWM-Robot && git fetch && git checkout feat/vjepa2-perception && git pull)
%cd /content/LeWM-Robot/vjepa-perception

In [ ]:
# Training/demo only need torch + safetensors + matplotlib (already on Colab).
# Skip the heavy install (no need to reload lerobot/transformers here).
!pip install -q safetensors matplotlib

## 3. Point at the cache on Drive

Edit the path below if your cache is elsewhere. The smoke-test notebook suggested `cp -r /content/cached_features /content/drive/MyDrive/`.

In [ ]:
CACHE_DIR = '/content/drive/MyDrive/cached_features/libero_object_smoke'

import json, pathlib
meta = json.loads((pathlib.Path(CACHE_DIR) / 'metadata.json').read_text())
print(f"episodes: {meta['num_episodes']}  frames: {meta['num_frames']}  encoder_dim: {meta['encoder_dim']}")

## 4. Train the BC head

With only 2 episodes (the smoke-test cache), 1 is held out for val. The model will overfit hard — the goal here is to validate the loop, not to win. Re-run the smoke-test notebook with `--max_episodes 10 --frame_stride 2` to get a more convincing demo.

In [ ]:
!python train.py \
    --cache_dir "$CACHE_DIR" \
    --output_dir /content/checkpoints/bc_smoke \
    --val_episodes 1 \
    --epochs 30 \
    --batch_size 32 \
    --lr 1e-3

## 5. Generate the demo plot

In [ ]:
!python demo.py \
    --cache_dir "$CACHE_DIR" \
    --checkpoint /content/checkpoints/bc_smoke/best.pt \
    --output /content/demo.png

In [ ]:
from IPython.display import Image
Image('/content/demo.png')

## 6. (Optional) Save checkpoint + demo back to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/lewm_robot_demo
!cp /content/checkpoints/bc_smoke/best.pt /content/drive/MyDrive/lewm_robot_demo/
!cp /content/checkpoints/bc_smoke/history.json /content/drive/MyDrive/lewm_robot_demo/
!cp /content/demo.png /content/drive/MyDrive/lewm_robot_demo/
!ls -la /content/drive/MyDrive/lewm_robot_demo/